In [24]:
import os
from dotenv import load_dotenv
from agents import Agent, Runner, trace, set_default_openai_api, set_default_openai_client, set_tracing_export_api_key
from agents.mcp import MCPServerStdio, create_static_tool_filter
from IPython.display import display, Markdown
from openai import AsyncOpenAI
from pathlib import Path
from datetime import datetime


load_dotenv(override=True)

# Model calls → the LiteLLM proxy
gateway = AsyncOpenAI(base_url=os.environ["OPENAI_BASE_URL"], api_key=os.environ["OPENAI_API_KEY"])
set_default_openai_client(gateway, use_for_tracing=False)   # don't use the LiteLLM key for traces
set_default_openai_api("chat_completions")

# Traces → OpenAI, with a real OpenAI key
set_tracing_export_api_key(os.environ["OPENAI_TRACING_API_KEY"])

model = "gpt-4o-mini"


In [9]:
from backend.accounts import Account

In [10]:
account = Account.get("Ed")
account.reset()
account

Account(name='ed', balance=10000.0, strategy='', holdings={}, transactions=[], portfolio_value_time_series=[])

In [11]:
account.buy_shares("AMZN", 3, "Because i love amazon")
account.report()

'{"name": "ed", "balance": 9233.56018, "strategy": "", "holdings": {"AMZN": 3}, "transactions": [{"symbol": "AMZN", "quantity": 3, "price": 255.47994, "timestamp": "2026-11-19 16:55:37", "rationale": "Because i love amazon"}], "portfolio_value_time_series": [["2026-11-19 16:55:37", 9998.47018], ["2026-11-19 16:55:37", 9998.47018]], "total_portfolio_value": 9998.47018, "total_profit_loss": -1.529819999999745}'

# MCP

Market Place: https://www.smithery.ai/ https://glama.ai/mcp/servers/upstash/context7


In [12]:
params = {"command": "uv", "args": ["run", "-m", "mcp_servers.accounts_server"]}
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as server:
    mcp_tools = await server.list_tools()

mcp_tools

[Tool(name='get_balance', title=None, description='Get the cash balance of the given account name.\n\nArgs:\n    name: The name of the account holder\n', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_balanceArguments', 'type': 'object'}, outputSchema={'properties': {'result': {'title': 'Result', 'type': 'number'}}, 'required': ['result'], 'title': 'get_balanceOutput', 'type': 'object'}, icons=None, annotations=None, meta=None, execution=None),
 Tool(name='get_holdings', title=None, description='Get the holdings of the given account name.\n\nArgs:\n    name: The name of the account holder\n', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_holdingsArguments', 'type': 'object'}, outputSchema={'additionalProperties': {'type': 'integer'}, 'title': 'get_holdingsDictOutput', 'type': 'object'}, icons=None, annotations=None, meta=None, execution=None),
 Tool(name='buy_s

In [14]:
instructions = "You are able to manage an account for a client, and answer questions about the account."
request = "My name is Ed and my account is under the name Ed. What's my balance and my holdings?"
model = "gpt-5.4-mini"
params = {"command": "uv", "args": ["run", "-m", "mcp_servers.accounts_server"]}

async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="account_manager", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("account_manager"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

Balance: $9,233.56  
Holdings: AMZN 3 shares

# Long Term Memory

In [16]:
memory_path = os.path.abspath("memory/memory.json")
memory_params = {"command": "npx", "args": ["-y", "@modelcontextprotocol/server-memory"], "env": {"MEMORY_FILE_PATH": memory_path}}

async with MCPServerStdio(params=memory_params, client_session_timeout_seconds=60) as server:
    memory_tools = await server.list_tools()

memory_tools

[Tool(name='create_entities', title='Create Entities', description='Create multiple new entities in the knowledge graph. An entity whose name already exists, or repeats an earlier entity in the same call, is skipped and its observations are not added; the result lists the skipped names in `skipped`. Use add_observations to add observations to an existing entity.', inputSchema={'type': 'object', '$schema': 'https://json-schema.org/draft/2020-12/schema', 'properties': {'entities': {'type': 'array', 'items': {'type': 'object', 'properties': {'name': {'type': 'string', 'description': 'The name of the entity'}, 'entityType': {'type': 'string', 'description': 'The type of the entity'}, 'observations': {'type': 'array', 'items': {'type': 'string'}, 'description': 'An array of observation contents associated with the entity'}}, 'required': ['name', 'entityType', 'observations']}, 'description': 'The entities to create, each with a name unique in the graph'}}, 'required': ['entities']}, outputS

In [17]:
instructions = "You use your entity tools as a persistent memory to store and recall information about your conversations."
request = "My name's Miguel. I'm an LLM engineer. I'm teaching a course about AI Agents, including the incredible MCP protocol. MCP is a protocol for connecting agents with tools, resources and prompt templates, and makes it easy to integrate AI agents with capabilities."
model = "gpt-5.4-mini"

First Call

In [22]:
memory_path = os.path.abspath("memory/memory.json")
memory_params = {"command": "npx", "args": ["-y", "@modelcontextprotocol/server-memory"], "env": {"MEMORY_FILE_PATH": memory_path}}

async with MCPServerStdio(params=memory_params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

Nice to meet you, Miguel — sounds like a great course.

Second Call

In [23]:
async with MCPServerStdio(params=memory_params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, "My name's Miguel. What do you know about me?")
    display(Markdown(result.final_output))

I know that you’re Miguel, an LLM engineer, and that you teach a course about AI Agents and the MCP protocol.

# RAG

In [25]:
vectordb_path = Path("memory/qdrant")
vectorstore_params = {
    "command": "uvx",
    "args": ["mcp-server-qdrant"],
    "env": {
        "QDRANT_LOCAL_PATH": str(vectordb_path),
        "COLLECTION_NAME": "knowledge",
    },
}

async with MCPServerStdio(params=vectorstore_params, client_session_timeout_seconds=120) as server:
    vectorstore_tools = await server.list_tools()

vectorstore_tools

[Tool(name='qdrant-find', title=None, description='Look up memories in Qdrant. Use this tool when you need to: \n - Find memories by their content \n - Access memories for further analysis \n - Get some personal information about the user', inputSchema={'properties': {'query': {'description': 'What to search for', 'title': 'Query', 'type': 'string'}}, 'required': ['query'], 'type': 'object'}, outputSchema=None, icons=None, annotations=None, meta=None, execution=None),
 Tool(name='qdrant-store', title=None, description='Keep the memory for later use, when you are asked to remember something.', inputSchema={'properties': {'information': {'description': 'Text to store', 'title': 'Information', 'type': 'string'}, 'metadata': {'anyOf': [{'additionalProperties': True, 'type': 'object'}, {'type': 'null'}], 'default': None, 'description': 'Extra metadata stored along with memorised information. Any json is accepted.', 'title': 'Metadata'}}, 'required': ['information'], 'type': 'object'}, outpu

In [ ]:
massive_api_key = os.getenv("MASSIVE_API_KEY")

if massive_api_key:
    market_params = {
        "command": "uvx",
        # mcp_massive still imports mcp.server.fastmcp, an API the mcp SDK's 2.0.0 release removed.
        # mcp_massive doesn't cap its mcp dependency, so pin uvx to an older, compatible mcp here.
        "args": ["--with", "mcp<2.0.0", "--from", "git+https://github.com/massive-com/mcp_massive@v0.10.0", "mcp_massive"],
        "env": {"MASSIVE_API_KEY": massive_api_key},
    }
else:
    market_params = {"command": "uv", "args": ["run", "-m", "mcp_servers.market_server"]}

async with MCPServerStdio(params=market_params, client_session_timeout_seconds=120) as server:
    market_tools = await server.list_tools()

market_tools

[Tool(name='search_endpoints', title=None, description='Search for market data API endpoints and built-in finance functions by natural language query. Use this FIRST to find the right endpoint before calling call_api. Covers stocks, options, forex, crypto, futures, indices, ETFs, and economic data. Pass market to pin results to a specific asset class when you already know it; omit it and the server will infer from the query. Use detail="more" to see query parameter docs needed for building call_api requests.', inputSchema={'additionalProperties': False, 'properties': {'query': {'description': 'Natural language search query for API endpoints', 'minLength': 1, 'title': 'Query', 'type': 'string'}, 'scope': {'anyOf': [{'enum': ['all', 'endpoints', 'functions'], 'type': 'string'}, {'type': 'null'}], 'default': None, 'description': 'Search scope: "endpoints" for API only, "functions" for local functions only, or "all"/omit for both', 'title': 'Scope'}, 'max_results': {'anyOf': [{'maximum': 2

In [ ]:
vectordb_path = Path("memory/qdrant")
vectorstore_params = {
    "command": "uvx",
    "args": ["mcp-server-qdrant"],
    "env": {
        "QDRANT_LOCAL_PATH": str(vectordb_path),
        "COLLECTION_NAME": "knowledge",
    },
}

massive_api_key = os.getenv("MASSIVE_API_KEY")

if massive_api_key:
    market_params = {
        "command": "uvx",
        # mcp_massive still imports mcp.server.fastmcp, an API the mcp SDK's 2.0.0 release removed.
        # mcp_massive doesn't cap its mcp dependency, so pin uvx to an older, compatible mcp here.
        "args": ["--with", "mcp<2.0.0", "--from", "git+https://github.com/massive-com/mcp_massive@v0.10.0", "mcp_massive"],
        "env": {"MASSIVE_API_KEY": massive_api_key},
    }
else:
    market_params = {"command": "uv", "args": ["run", "-m", "mcp_servers.market_server"]}

INSTRUCTIONS = """You are a senior stock analyst and an expert on the US stock market. You answer questions about it.
Use the market tools for prices and market data.
When you learn something worth keeping, store it in your knowledge base (qdrant-store), with its date and source.
When you're asked what you know, search your knowledge base (qdrant-find) first and answer from it."""

request = "What was the latest available historical close for Apple (AAPL)?"

model = "gpt-5.4-mini"

async with MCPServerStdio(params=market_params, client_session_timeout_seconds=120) as search_server:
    async with MCPServerStdio(params=vectorstore_params, client_session_timeout_seconds=120) as vector_server:
        agent = Agent(name="researcher", instructions=INSTRUCTIONS, model=model, mcp_servers=[search_server, vector_server])
        with trace("research and store"):
            result = await Runner.run(agent,request, max_turns=20)
        display(Markdown(result.final_output))
        for item in result.new_items:
            if item.type == "tool_call_item":
                print("🔧", item.raw_item.name, item.raw_item.arguments)
            elif item.type == "tool_call_output_item":
                print("   →", str(item.output)[:200])

AAPL’s latest available daily close was **$352.45**.

In [35]:
vectordb_path = Path("memory/qdrant")
vectorstore_params = {
    "command": "uvx",
    "args": ["mcp-server-qdrant"],
    "env": {
        "QDRANT_LOCAL_PATH": str(vectordb_path),
        "COLLECTION_NAME": "knowledge",
    },
}

async with MCPServerStdio(params=vectorstore_params, client_session_timeout_seconds=120) as vector_server:
    agent = Agent(name="researcher", instructions=INSTRUCTIONS, model=model, mcp_servers=[vector_server])
    with trace("retrieve"):
        result = await Runner.run(agent, "Based on your knowledge base, what's the latest on the Apple (AAPL) stock?")
    display(Markdown(result.final_output))

I don’t have any saved Apple/AAPL notes in my knowledge base yet.

If you want, I can still give you a current-style overview, or you can ask me to store a specific Apple fact/update for later.